In [1]:
from parsers.ll import LL1Parser

## Exercise 3.4

<img src="static/exercise_3_4.png" alt="Exercise 3.4 from book" width="600">

This exercise is given to the grammar 3.1, which is defined here:

<img src="static/grammar_3_1.png" alt="Grammar 3.1" width="600">

The end goal of this exercise is probably LL(1) grammar which don't produce
more than one rule per parsing table element, so I will define it and check
the resulting parsing table.

first, let's see parsing table without modyfing the grammar.

In [2]:
naive_p = LL1Parser()
naive_p.add_rules(
    "S -> S ; S",
    "S -> id := E",
    "S -> print ( L )",
    "E -> id",
    "E -> num",
    "E -> E + E",
    "E -> ( S , E )",
    "L -> E",
    "L -> L , E",
)

In [3]:
print(naive_p)

S─┬─ S ; S
  ├─ id := E
  └─ print ( L )
E─┬─ id
  ├─ num
  ├─ E + E
  └─ ( S , E )
L─┬─ E
  └─ L , E



In [4]:
print(naive_p.get_tabulate())

┌────┬────────────────┬─────┬─────┬─────┬──────┬─────┬──────────────┬────────────┬──────────────────┐
│    │ (              │ )   │ +   │ ,   │ :=   │ ;   │ id           │ num        │ print            │
├────┼────────────────┼─────┼─────┼─────┼──────┼─────┼──────────────┼────────────┼──────────────────┤
│ E  │ E -> E + E     │     │     │     │      │     │ E -> id      │ E -> num   │                  │
│    │ E -> ( S , E ) │     │     │     │      │     │ E -> E + E   │ E -> E + E │                  │
├────┼────────────────┼─────┼─────┼─────┼──────┼─────┼──────────────┼────────────┼──────────────────┤
│ L  │ L -> E         │     │     │     │      │     │ L -> E       │ L -> E     │                  │
│    │ L -> L , E     │     │     │     │      │     │ L -> L , E   │ L -> L , E │                  │
├────┼────────────────┼─────┼─────┼─────┼──────┼─────┼──────────────┼────────────┼──────────────────┤
│ S  │                │     │     │     │      │     │ S -> id := E │            │

We clearly see that there are some cells in the table with more than one value.
This means that current state of this grammar is not parsable using LL(1) parsing.

### Left recursion elimination method

Let's try formula, to eliminate left recursion from grammar, found at the book.

$$
\begin{pmatrix}
X \to X \gamma_1 \\
X \to X \gamma_2 \\
X \to \alpha_1 \\
X \to \alpha_2
\end{pmatrix}
\implies
\begin{pmatrix}
X \to \alpha_1 X' \\
X \to \alpha_2 X' \\
X' \to \gamma_1 X' \\
X' \to \gamma_2 X' \\
X' \to \epsilon
\end{pmatrix}
$$

In [5]:
solved_p = LL1Parser()

solved_S_rules = [
    "Start -> S S' &",
    "S -> id := E",
    "S -> print ( L )",
    "S' -> ; S",
    "S' -> "
]

solved_E_rules = [
    "D -> id",
    "D -> num",
    "E -> D E'",
    "E' -> ",
    "E' -> + E",
    "E -> ( S , E )",
]

solved_p.add_rules(
    *solved_S_rules,
    *solved_E_rules,
    "L -> E LE",
    "LE -> , E",
    "LE -> "
)
print(solved_p.get_tabulate())

┌───────┬─────────┬────────────────┬─────────┬───────────┬───────────┬──────┬───────────┬─────────────────┬───────────┬──────────────────┐
│       │ &       │ (              │ )       │ +         │ ,         │ :=   │ ;         │ id              │ num       │ print            │
├───────┼─────────┼────────────────┼─────────┼───────────┼───────────┼──────┼───────────┼─────────────────┼───────────┼──────────────────┤
│ D     │         │                │         │           │           │      │           │ D -> id         │ D -> num  │                  │
├───────┼─────────┼────────────────┼─────────┼───────────┼───────────┼──────┼───────────┼─────────────────┼───────────┼──────────────────┤
│ E     │         │ E -> ( S , E ) │         │           │           │      │           │ E -> D E'       │ E -> D E' │                  │
├───────┼─────────┼────────────────┼─────────┼───────────┼───────────┼──────┼───────────┼─────────────────┼───────────┼──────────────────┤
│ E'    │ E' -> ε │        

In [6]:
print(solved_p)

Start─── S S' &
S─────┬─ id := E
      └─ print ( L )
S'────┬─ ; S
      └─ ε
D─────┬─ id
      └─ num
E─────┬─ D E'
      └─ ( S , E )
E'────┬─ ε
      └─ + E
L─────── E LE
LE────┬─ , E
      └─ ε



## Exercise 3.5

In [7]:
p = LL1Parser()
p.add_rules(
    "S` -> S &",
    "S -> ",
    "S -> X S",
    r"B -> \ begin { WORD }",
    r"E -> \ end { WORD }",
    "X -> B S E",
    "X -> { S }",
    "X -> WORD",
    "X -> begin",
    "X -> end",
    r"X -> \ WORD",
)

In [8]:
p.terminals

{'&', 'WORD', '\\', 'begin', 'end', '{', '}'}

In [9]:
p.non_terminals

{'B', 'E', 'S', 'S`', 'X'}

In [10]:
p.compute_first_follow_nullable()

In [11]:
print(p)

S`─── S &
S──┬─ ε
   └─ X S
B──── \ begin { WORD }
E──── \ end { WORD }
X──┬─ B S E
   ├─ { S }
   ├─ WORD
   ├─ begin
   ├─ end
   └─ \ WORD



In [12]:
{k: v for k, v in p.first.items() if k in p.non_terminals}

{'S`': {'&', 'WORD', '\\', 'begin', 'end', '{'},
 'S': {'WORD', '\\', 'begin', 'end', '{'},
 'X': {'WORD', '\\', 'begin', 'end', '{'},
 'B': {'\\'},
 'E': {'\\'}}

In [13]:
p.follow["S"]

{'&', '\\', '}'}

In [14]:
print(p.get_tabulate(fmt='simple_grid'))

┌────┬───────────┬───────────┬───────────────────────┬────────────┬───────────┬────────────┬────────┐
│    │ &         │ WORD      │ \                     │ begin      │ end       │ {          │ }      │
├────┼───────────┼───────────┼───────────────────────┼────────────┼───────────┼────────────┼────────┤
│ B  │           │           │ B -> \ begin { WORD } │            │           │            │        │
├────┼───────────┼───────────┼───────────────────────┼────────────┼───────────┼────────────┼────────┤
│ E  │           │           │ E -> \ end { WORD }   │            │           │            │        │
├────┼───────────┼───────────┼───────────────────────┼────────────┼───────────┼────────────┼────────┤
│ S  │ S -> ε    │ S -> X S  │ S -> X S              │ S -> X S   │ S -> X S  │ S -> X S   │ S -> ε │
│    │           │           │ S -> ε                │            │           │            │        │
├────┼───────────┼───────────┼───────────────────────┼────────────┼───────────┼───

## Exercise 3.6

<img src="static/exercise_3_6.png" alt="Grammar 3.1" width="600">


In [15]:
p = LL1Parser()
p.add_rules(
    "S -> u B D z",
    "B -> B v",
    "B -> w",
    "D -> E F",
    "E -> y",
    "E -> ",
    "F -> x",
    "F -> ",
)

### A

first, follow, nullable

In [16]:
p.compute_first_follow_nullable()
print("first")
print("\n".join(f"{k} -> {v}" for k, v in p.first.items() if k in p.non_terminals))
print("follows")
print("\n".join(f"{k} -> {v}" for k, v in p.follow.items() if k in p.non_terminals))
print("nullables")
print(p.nullables)

first
S -> {'u'}
B -> {'w'}
D -> {'y', 'x'}
E -> {'y'}
F -> {'x'}
follows
B -> {'v', 'y', 'x', 'z'}
D -> {'z'}
S -> set()
E -> {'x', 'z'}
F -> {'z'}
nullables
{'E', 'D', 'F'}


### B

Construct parsing table

In [17]:
print(p.get_tabulate())

┌────┬──────────────┬─────┬──────────┬──────────┬──────────┬──────────┐
│    │ u            │ v   │ w        │ x        │ y        │ z        │
├────┼──────────────┼─────┼──────────┼──────────┼──────────┼──────────┤
│ B  │              │     │ B -> B v │          │          │          │
│    │              │     │ B -> w   │          │          │          │
├────┼──────────────┼─────┼──────────┼──────────┼──────────┼──────────┤
│ D  │              │     │          │ D -> E F │ D -> E F │ D -> E F │
├────┼──────────────┼─────┼──────────┼──────────┼──────────┼──────────┤
│ E  │              │     │          │ E -> ε   │ E -> y   │ E -> ε   │
├────┼──────────────┼─────┼──────────┼──────────┼──────────┼──────────┤
│ F  │              │     │          │ F -> x   │          │ F -> ε   │
├────┼──────────────┼─────┼──────────┼──────────┼──────────┼──────────┤
│ S  │ S -> u B D z │     │          │          │          │          │
└────┴──────────────┴─────┴──────────┴──────────┴──────────┴────

### C

This grammar isn't LL(1) compatible because:
* There is left recursion for B non terminal symbol
* Prasing table shows conflict for B

### D

change as little as possible to make the grammar LL(1) parsable.

In [19]:
p = LL1Parser()
p.add_rules(
    "S -> u B D z",
    "B -> w B'",
    "B' -> v B'",
    "D -> E F",
    "E -> y",
    "E -> ",
    "F -> x",
    "F -> ",
)
print(p.get_tabulate())

┌────┬──────────────┬────────────┬───────────┬──────────┬──────────┬──────────┐
│    │ u            │ v          │ w         │ x        │ y        │ z        │
├────┼──────────────┼────────────┼───────────┼──────────┼──────────┼──────────┤
│ B  │              │            │ B -> w B' │          │          │          │
├────┼──────────────┼────────────┼───────────┼──────────┼──────────┼──────────┤
│ B' │              │ B' -> v B' │           │          │          │          │
├────┼──────────────┼────────────┼───────────┼──────────┼──────────┼──────────┤
│ D  │              │            │           │ D -> E F │ D -> E F │ D -> E F │
├────┼──────────────┼────────────┼───────────┼──────────┼──────────┼──────────┤
│ E  │              │            │           │ E -> ε   │ E -> y   │ E -> ε   │
├────┼──────────────┼────────────┼───────────┼──────────┼──────────┼──────────┤
│ F  │              │            │           │ F -> x   │          │ F -> ε   │
├────┼──────────────┼────────────┼──────

Used classic techinque for removing left recursion. Thanks to this I've got one additional rule,
but whole grammar now is parsable by LL1 parser.

## Grammar 3.15

I want just to check if my implementation of LL(1) parsing.

In [ ]:
g3_15 = LL1Parser()
g3_15.add_rules(
    "S -> E $",
    "T -> F T'",
    "E -> T E'",
    "E' -> + T E'",
    "E' -> - T E'",
    "E' ->",
    "T' -> * F T'",
    "T' -> / F T'",
    "T' ->",
    "F -> id",
    "F -> num",
    "F -> ( E )",
)
print(g3_15)

In [ ]:
print(g3_15.get_tabulate())

Original grammar table from the book.

<img src="static/grammar_3_15.png" alt="Grammar 3.15" width="600">

Looks like every field is the same in both tables, so I assume that my implementation
of LL(1) algorithm is okay.

## Neso academy example

Let's check how it work in the example, where we can be sure that's working case

[video link](https://www.youtube.com/watch?v=DT-cbznw9aY)

In [ ]:
neso_grm = LL1Parser()

In [ ]:
neso_grm.add_rules(
    r"E -> T E' &",
    r"E' -> + T E'",
    r"E' -> ",
    r"T -> F T'",
    r"T' -> * F T'",
    r"T' -> ",
    r"F -> id",
    r"F -> ( E )",
)

In [ ]:
neso_grm.non_terminals

In [ ]:
neso_grm.terminals

In [ ]:
neso_grm.compute_first_follow_nullable()
for k, v in neso_grm.first.items():
    if k in neso_grm.non_terminals:
        print(f"{k} {v}")

In [ ]:
for k, v in neso_grm.follow.items():
    if k in neso_grm.non_terminals:
        print(f"{k} {v}")

In [ ]:
print(neso_grm)

In [ ]:
print(neso_grm.get_tabulate())